# Example 00 Lecture Note: One Object to One Feature Row

This notebook is a first-principles walkthrough from raw data to a single feature vector.

## Learning goals
- Understand the role of `ConstructionOptions`, `FiltrationSpec`, and `PipelineOptions`.
- See exactly what `encode(...; degree=0)` returns and how to inspect it.
- Run sanity invariants before featurization.
- Export reproducible artifacts for downstream analysis.

## Mathematical flow
Raw point cloud -> filtered complex -> encoded module `H^0` over a finite poset -> invariant summaries -> feature vector.
        


## Why these choices?

For a first notebook we intentionally pick conservative settings:
- `kind=:rips_density`: canonical two-parameter point-cloud filtration.
- `max_dim=0`: fastest possible first pass.
- `sparsify=:knn`: avoid dense all-pairs growth.
- `field=F2()`: deterministic and fast linear algebra.

Reasonable alternatives after the first run:
- `max_dim=1` to include edge-level topology.
- `kind=:function_rips` with an explicit vertex function.
- `field=QQ()` if exact arithmetic is preferred over speed.

### About `output_stage` vs `stage`
You can request a target stage in two places:
- `ConstructionOptions(output_stage=...)` sets the default stage policy.
- `encode(...; stage=...)` sets the stage for this specific call.

Canonical stage set:
- `:auto`
- `:simplex_tree`
- `:graded_complex`
- `:cochain`
- `:module`
- `:fringe`
- `:flange`
- `:encoding_result`

Rule of thumb:
- Keep `stage=:auto` for onboarding and set `construction.output_stage` once.
- Override `stage` directly only when you want a one-off diagnostic return object.

Practical interpretation:
- Use early stages (`:simplex_tree`, `:graded_complex`) for ingestion diagnostics/benchmarking.
- Use `:encoding_result` for normal invariant + feature workflows.


In [ ]:
# Load shared example helpers.
common_candidates = (
    joinpath(pwd(), "examples", "_common.jl"),
    joinpath(pwd(), "_common.jl"),
)

common_path = nothing
for p in common_candidates
    if isfile(p)
        common_path = p
        break
    end
end
common_path === nothing && error("Could not find examples/_common.jl. Start from repo root or examples/.")
include(common_path)

println("Loaded _common.jl from: ", common_path)

Random.seed!(20260217)
outdir = example_outdir("00_quickstart_notebook")
println("outdir = ", outdir)


## Step 1: Build one deterministic point cloud

We use a tiny noisy circle so every downstream object is easy to inspect.
        


In [ ]:
# Use shared deterministic point-cloud helper from _common.jl
cloud = make_noisy_circle_cloud(12; noise=0.03, seed=11)
println("cloud type      = ", typeof(cloud))
println("number of points= ", length(cloud.points))
println("first point     = ", cloud.points[1])


## Step 2: Declare ingestion and encoding contracts

Interpretation of key knobs:
- `sparsify=:knn`: build a sparse neighborhood graph first.
- `collapse=:none`: disable graph/complex collapse for this didactic run.
- `budget=(...)`: hard caps (simplices, edges, bytes) to prevent accidental blowups.
- `output_stage=:encoding_result`: canonical output for downstream invariants/features.
- `axes_policy=:encoding`: keep downstream invariant/feature axes aligned to the
  axes induced by the encoding itself.

### What does `axes_policy=:encoding` mean concretely?
It means: when an invariant/featurizer needs axis grids, it uses the axis system
coming from the encoded object (`enc`) instead of requiring a separate manual axis
specification in every call.

Why this is useful in onboarding:
- fewer knobs to remember,
- lower risk of accidental axis mismatch,
- output is easier to interpret because encoding and feature geometry agree.

When might you choose a different policy?
- If you need fixed externally-defined grids for cross-experiment comparability,
  use explicit axis settings in `InvariantOptions` (for example, `axes_policy=:as_given`).


In [ ]:
construction = PM.ConstructionOptions(
    ;
    sparsify=:knn,
    collapse=:none,
    output_stage=:encoding_result,
    budget=(max_simplices=80_000, max_edges=20_000, memory_budget_bytes=600_000_000),
)

filtration = PM.FiltrationSpec(
    ;
    kind=:rips_density,
    max_dim=0,
    knn=4,
    density_k=4,
    nn_backend=:auto,
)

pipeline = PM.PipelineOptions(
    ;
    orientation=(1, 1),
    axes_policy=:encoding,
    axis_kind=:rn,
    poset_kind=:signature,
)

field = PM.CoreModules.F2()

println("construction type = ", typeof(construction))
println("filtration type   = ", typeof(filtration))
println("pipeline type     = ", typeof(pipeline))
println("field type        = ", typeof(field))


## Step 3: Encode and inspect `EncodingResult`

`degree=t` means "return the cohomology module `H^t`" from the constructed cochain complex.

Here we use `degree=0`, so we are inspecting `H^0`.

`cache=:auto` means per-call automatic caching; it improves repeated work *within*
this call without requiring explicit `SessionCache` management.

`EncodingResult` exposes three core objects:
- `enc.P`: finite poset,
- `enc.M`: module over that poset,
- `enc.pi`: classifier map from raw grades to poset vertices.

Call degree hom_degree. Distinguish the various meanings of degree.


In [ ]:
enc = PM.encode(
    cloud,
    filtration;
    degree=0,
    field=field,
    cache=:auto,
    stage=:auto,
    construction=construction,
    pipeline=pipeline,
)

println("enc type   = ", typeof(enc))
println("enc.M type = ", typeof(enc.M))
println("enc.pi type= ", typeof(enc.pi))
println("|P|        = ", length(enc.M.dims))

# Coarsening is optional; here it keeps vectors small and readable.
# For pairwise-comparison workflows, coarsen jointly so both modules share a compatible grid.
enc_small = PM.coarsen(enc; method=:uptight)
println("|P| after coarsen = ", length(enc_small.M.dims))


## Step 4: Sanity invariants before featurization

We check two basic summaries:
- `rank_invariant` (sparse rank table over comparable pairs),
- `restricted_hilbert` (dimension per poset vertex).

`InvariantOptions(pl_mode=:fast)` uses the strict PL mode contract (`:fast` or `:verified`).


In [ ]:
opts = PM.InvariantOptions(
    ;
    axes_policy=:encoding,
    max_axis_len=48,
    threads=false,
    strict=true,
    pl_mode=:fast,
)

rank_tbl = PM.rank_invariant(enc_small; opts=opts, store_zeros=false)
rh = PM.restricted_hilbert(enc_small.M)

println("rank entries (nonzero) = ", length(rank_tbl))
println("restricted_hilbert nnz = ", count(!iszero, rh))


## Step 5: Freeze a canonical vectorization

A **featurizer spec** is a typed contract that defines a deterministic map

`Phi_spec : (encoded object) -> R^d`

including:
- what quantity is sampled (here: rank-grid values),
- where it is sampled (grid/vertex layout),
- how it is represented (dense vs sparse / include zeros or not),
- output dimension `d` and feature naming.

For this notebook, `RankGridSpec(...)` defines `Phi_spec` on the encoded module.
Each coordinate corresponds to one **rank-grid entry**.

Here, a rank-grid entry is defined by a comparable pair `(u, v)` with `u <= v`,
and its value is `rank(M(u -> v))`, i.e. the rank of the module map from `u` to `v`.

`transform(spec, enc_small; ...)` is the *evaluation* of that map on one object:
it computes the numeric vector `Phi_spec(enc_small)`.

`nfeatures(spec)` reports `d`, and `feature_names(spec)` gives the coordinate labels.
This is what makes the output stable and statistics-ready across runs.


In [ ]:
spec = PM.RankGridSpec(nvertices=length(enc_small.M.dims), store_zeros=false, threads=false)
# Evaluate Phi_spec(enc_small) to get one fixed-length feature vector.
vec = PM.transform(spec, enc_small; opts=opts, threaded=false)

println("nfeatures(spec) = ", PM.nfeatures(spec))
println("vector length   = ", length(vec))

# Wrap one vector into a one-row FeatureSet so export is identical to batch workflows.
fs = PM.FeatureSet(
    reshape(vec, 1, :),
    PM.feature_names(spec),
    ["quickstart_001"],
    (label=["circle"], note="example_00_notebook"),
)


## Step 6: Export feature tables and reproducibility artifacts

We use the canonical `save_features(...)` API in both wide and long modes.
Then we save dataset and pipeline JSON so this run is replayable.


In [ ]:
wide_path = joinpath(outdir, "quickstart__wide.csv")
long_path = joinpath(outdir, "quickstart__long.csv")

PM.save_features(wide_path, fs; format=:csv, mode=:wide, metadata=true)
PM.save_features(long_path, fs; format=:csv, mode=:long, metadata=true)

dset_path = joinpath(outdir, "dataset.json")
pipe_path = joinpath(outdir, "pipeline.json")

PM.save_dataset_json(dset_path, cloud)
PM.save_pipeline_json(
    pipe_path,
    cloud,
    filtration;
    degree=0,
    pipeline_opts=PM.PipelineOptions(
        orientation=pipeline.orientation,
        axes_policy=pipeline.axes_policy,
        axis_kind=pipeline.axis_kind,
        poset_kind=pipeline.poset_kind,
        field=:F2,
    ),
)

println("wide csv      = ", wide_path)
println("long csv      = ", long_path)
println("dataset json  = ", dset_path)
println("pipeline json = ", pipe_path)


## Discussion prompts
- What changed when coarsening was applied?
- Which choice most strongly controls runtime here (`max_dim`, `sparsify`, or `knn`)?
- If you switch to `QQ()`, which parts of the run should you expect to slow down first?
- How does changing `knn` to another option affect the output
        
